
# NeoLM — Prepare Token Streams

Convert NeoCorpus into binary token streams for efficient language model pretraining.

## Pipeline

1. Load NeoCorpus and the trained NeoLM tokenizer.
2. Tokenize documents and append EOS tokens.
3. Save train, validation, and test streams as uint16 binary files.
4. Validate token counts and training block boundaries.

## Configuration

- Vocabulary: 49,152
- Context length: 1,024
- Stride: 1,023
- Storage: NumPy binary files (`.bin`)
- Training access: Memory-mapped token streams

Documents are concatenated with EOS separators. Training blocks are constructed from the resulting continuous streams.


In [ ]:
from pathlib import Path

import numpy as np
from datasets import load_from_disk
from transformers import AutoTokenizer
from tqdm.auto import tqdm

# Paths
DB_DIR = Path("../../artifacts/pretrain_db/final")
MODEL_DIR = Path("../../artifacts/model")
STREAM_DIR = Path("../../artifacts/pretrain_db")

# Configuration
BLOCK_SIZE = 1024
STRIDE = BLOCK_SIZE - 1
TOKEN_DTYPE = np.uint16
BATCH_SIZE = 1000

# Load assets
pretrain_db = load_from_disk(str(DB_DIR))

tokenizer = AutoTokenizer.from_pretrained(
    str(MODEL_DIR),
    use_fast=True,
    local_files_only=True,
)

assert len(tokenizer) == 49_152
assert tokenizer.eos_token_id == 0
assert len(tokenizer) <= np.iinfo(TOKEN_DTYPE).max + 1

print("=" * 60)
print("NEOLM — TOKEN STREAM PREPARATION")
print("=" * 60)
print(f"Train documents: {len(pretrain_db['train']):,}")
print(f"Val documents:   {len(pretrain_db['validation']):,}")
print(f"Test documents:  {len(pretrain_db['test']):,}")
print(f"Vocabulary:      {len(tokenizer):,}")
print(f"Block size:      {BLOCK_SIZE:,}")
print(f"Stride:          {STRIDE:,}")
print(f"Token dtype:     {np.dtype(TOKEN_DTYPE)}")
print(f"Output:          {STREAM_DIR}")
print("=" * 60)

In [ ]:
def iter_documents(dataset):
    for row in dataset:
        text = row["text"].strip()

        if text:
            yield text


def build_token_stream(dataset, output_path, total_docs=None):
    eos_id = tokenizer.eos_token_id
    backend = tokenizer.backend_tokenizer
    reserved = (tokenizer.eos_token, tokenizer.pad_token)

    total_tokens = 0
    written_docs = 0
    skipped_docs = 0
    batch = []

    def write_batch(texts, dst):
        encoded = backend.encode_batch(
            texts,
            add_special_tokens=False,
        )

        size = sum(len(enc.ids) + 1 for enc in encoded)
        arr = np.empty(size, dtype=TOKEN_DTYPE)

        pos = 0

        for enc in encoded:
            ids = enc.ids
            n = len(ids)

            arr[pos:pos + n] = ids
            arr[pos + n] = eos_id
            pos += n + 1

        arr.tofile(dst)

        return len(encoded), size

    with open(output_path, "wb") as dst:
        for text in tqdm(
            iter_documents(dataset),
            total=total_docs,
            desc=f"Tokenizing {output_path.name}",
            unit="docs",
        ):
            if any(s in text for s in reserved if s):
                skipped_docs += 1
                continue

            batch.append(text)

            if len(batch) >= BATCH_SIZE:
                docs, tokens = write_batch(batch, dst)

                written_docs += docs
                total_tokens += tokens
                batch.clear()

        if batch:
            docs, tokens = write_batch(batch, dst)

            written_docs += docs
            total_tokens += tokens

    print(f"\n{'=' * 60}")
    print(f"TOKEN STREAM — {output_path.name}")
    print(f"{'=' * 60}")
    print(f"Documents: {written_docs:,}")
    print(f"Tokens:    {total_tokens:,}")
    print(f"Skipped:   {skipped_docs:,}")
    print(f"{'=' * 60}")

    return written_docs, total_tokens

In [ ]:
splits = {
    "train": pretrain_db["train"],
    "val": pretrain_db["validation"],
    "test": pretrain_db["test"],
}

for name, dataset in splits.items():
    bin_path = STREAM_DIR / f"{name}.bin"

    docs, tokens = build_token_stream(
        dataset,
        bin_path,
        total_docs=len(dataset),
    )

    assert docs == len(dataset)
    assert tokens >= BLOCK_SIZE

    blocks = (tokens - BLOCK_SIZE) // STRIDE + 1

    unused_tokens = tokens - (
        (blocks - 1) * STRIDE + BLOCK_SIZE
    )

    assert 0 <= unused_tokens < STRIDE

    print(f"\n{'=' * 60}")
    print(f"{name.upper()} STREAM")
    print(f"{'=' * 60}")
    print(f"Documents:     {docs:,}")
    print(f"Tokens:        {tokens:,}")
    print(f"Blocks:        {blocks:,}")
    print(f"Unused tokens: {unused_tokens:,}")
    print(f"File:          {bin_path}")
    print(f"{'=' * 60}")

print("\nTOKEN STREAMS READY ✅")

In [ ]:
for split in ("train", "val", "test"):
    path = STREAM_DIR / f"{split}.bin"

    tokens = np.memmap(
        path,
        dtype=TOKEN_DTYPE,
        mode="r",
    )

    n_blocks = (len(tokens) - BLOCK_SIZE) // STRIDE + 1

    first_block = tokens[:BLOCK_SIZE]
    last_start = (n_blocks - 1) * STRIDE
    last_block = tokens[last_start:last_start + BLOCK_SIZE]

    assert len(first_block) == BLOCK_SIZE
    assert len(last_block) == BLOCK_SIZE
    assert first_block.max() < len(tokenizer)
    assert last_block.max() < len(tokenizer)

    print(
        f"{split:5} | "
        f"Tokens: {len(tokens):>13,} | "
        f"Blocks: {n_blocks:>10,} | "
        f"Size: {path.stat().st_size / 1024**3:.2f} GiB"
    )

print("\nMEMMAP STREAMS VERIFIED ✅")